# Hierarchical Dataset Matrix Aggregation

Scans `runs/hier_dataset_matrix` for `loss_and_metrics.pkl` files, pulls out the metrics needed for the ablation matrix, and writes `aggregate_metrics.csv`. Adjust `ROOT_RESULTS_DIR` if your runs live elsewhere.

In [1]:
from pathlib import Path
import pickle
import numpy as np
import pandas as pd

ROOT_RESULTS_DIR = Path('runs/hier_dataset_matrix').resolve()
OUTPUT_CSV = ROOT_RESULTS_DIR / 'aggregate_metrics.csv'

if not ROOT_RESULTS_DIR.exists():
    raise FileNotFoundError(f'Expected {ROOT_RESULTS_DIR} to exist. Update ROOT_RESULTS_DIR if needed.')

print(f'Writing aggregate metrics to {OUTPUT_CSV}')

Writing aggregate metrics to /home/laura/Documents/Uni/Communication and Abstraction/emergent-abstractions-HW2/runs/hier_dataset_matrix/aggregate_metrics.csv


In [2]:
def parse_metadata(pkl_path: Path, root: Path):
    metadata = {}
    for part in pkl_path.relative_to(root).parts[:-1]:  # ignore filename
        if '=' not in part:
            continue
        key, value = part.split('=', 1)
        if key == 'seed':
            try:
                value = int(value)
            except ValueError:
                pass
        metadata[key] = value
    return metadata


def extract_message_lengths(blob):
    metrics = blob.get('metrics_test1', {})
    if not isinstance(metrics, dict):
        return []
    return [value for _, value in sorted(metrics.items())]

In [3]:
records = []
for pkl_path in sorted(ROOT_RESULTS_DIR.rglob('loss_and_metrics.pkl')):
    meta = parse_metadata(pkl_path, ROOT_RESULTS_DIR)
    with pkl_path.open('rb') as f:
        blob = pickle.load(f)

    msg_lengths = extract_message_lengths(blob)
    avg_len = float(np.mean(msg_lengths)) if msg_lengths else None
    final_len = float(msg_lengths[-1]) if msg_lengths else None

    record = {
        **meta,
        'final_test_acc': blob.get('final_test_acc'),
        'final_test_loss': blob.get('final_test_loss'),
        'avg_message_length': avg_len,
        'final_message_length': final_len,
        'result_path': str(pkl_path.parent),
    }
    records.append(record)

if not records:
    raise RuntimeError('No loss_and_metrics.pkl files found under the configured root.')

df = pd.DataFrame(records)
priority_cols = ['dataset', 'split', 'context', 'cost', 'seed']
ordered_cols = [col for col in priority_cols if col in df.columns] + [
    col for col in df.columns if col not in priority_cols
]
df = df[ordered_cols]
sort_cols = [col for col in priority_cols if col in df.columns]
if sort_cols:
    df = df.sort_values(by=sort_cols).reset_index(drop=True)

df.to_csv(OUTPUT_CSV, index=False)
print(f'Wrote {len(df)} rows to {OUTPUT_CSV}')
df.head()

Wrote 240 rows to /home/laura/Documents/Uni/Communication and Abstraction/emergent-abstractions-HW2/runs/hier_dataset_matrix/aggregate_metrics.csv


,dataset,split,context,cost,seed,dimensions,final_test_acc,final_test_loss,avg_message_length,final_message_length,result_path
0,flat,generic,shared,length_cost,0,"D(3,4)",0.958333,0.144966,3.853977,4.0,/home/laura/Documents/Uni/Communication and Ab...
1,flat,generic,shared,length_cost,0,"D(3,8)",0.507292,0.696186,1.000000,1.0,/home/laura/Documents/Uni/Communication and Ab...
2,flat,generic,shared,length_cost,0,"D(4,4)",0.505625,0.696358,1.000000,1.0,/home/laura/Documents/Uni/Communication and Ab...
3,flat,generic,shared,length_cost,1,"D(3,4)",0.989583,0.043729,3.859759,4.0,/home/laura/Documents/Uni/Communication and Ab...
4,flat,generic,shared,length_cost,1,"D(3,8)",0.493750,0.696225,1.000000,1.0,/home/laura/Documents/Uni/Communication and Ab...
